# Industrial Anomaly Detection
MVTec AD veri seti üzerinde veri inceleme, ROC/AUROC ve PyTorch temelleri.
Veri seti: MVTec AD (CC BY-NC-SA 4.0).

## Veri inceleme
MVTec AD klasör yapısı, train/good ve test sayıları, kusurlu görüntü ve maskesi.

In [ ]:
import os
base = "/kaggle/input/datasets/ipythonx"
for d in os.listdir(base):
    print(d)
    for sub in os.listdir(f"{base}/{d}")[:20]:
        print("   ", sub)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
CATEGORY = "metal_nut"

cat = ROOT / CATEGORY
print("train/good:", len(list((cat/"train/good").glob("*.png"))))
for d in sorted((cat/"test").iterdir()):
    print(f"test/{d.name}:", len(list(d.glob("*.png"))))

In [ ]:
import matplotlib.pyplot as plt
import cv2

def load(p):
    return cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)

files = sorted((cat/"train/good").glob("*.png"))[:6]
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, f in zip(axes, files):
    ax.imshow(load(f)); ax.axis("off")
plt.suptitle(f"{CATEGORY} - train/good")
plt.show()

In [ ]:
defect_type = [d.name for d in (cat/"test").iterdir() if d.name != "good"][0]
img_path  = sorted((cat/"test"/defect_type).glob("*.png"))[0]
mask_path = cat/"ground_truth"/defect_type/(img_path.stem + "_mask.png")

img  = load(img_path)
mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img);                axes[0].set_title(f"Kusurlu: {defect_type}")
axes[1].imshow(mask, cmap="gray");  axes[1].set_title("Maske")
axes[2].imshow(img); axes[2].imshow(mask, cmap="Reds", alpha=0.4); axes[2].set_title("Üst üste")
for a in axes: a.axis("off")
plt.show()

### Kategori görselleştirme (metal_nut ve leather)
Sayıları yazdırır, sağlam görüntüleri ve bir kusurlu görüntüyü maskesiyle gösterir.

In [ ]:
def goster(category, defect_type=None):
    cat = ROOT / category

    # 1) Sayılar
    print(f"--- {category} ---")
    print("train/good:", len(list((cat/"train/good").glob("*.png"))))
    for d in sorted((cat/"test").iterdir()):
        print(f"test/{d.name}:", len(list(d.glob("*.png"))))

    # 2) Sağlam eğitim görüntüleri
    files = sorted((cat/"train/good").glob("*.png"))[:6]
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    for ax, f in zip(axes, files):
        ax.imshow(load(f)); ax.axis("off")
    plt.suptitle(f"{category} - train/good")
    plt.show()

    # 3) Kusurlu görüntü + maske
    if defect_type is None:
        defect_type = sorted(d.name for d in (cat/"test").iterdir() if d.name != "good")[0]
    img_path  = sorted((cat/"test"/defect_type).glob("*.png"))[0]
    mask_path = cat/"ground_truth"/defect_type/(img_path.stem + "_mask.png")

    img  = load(img_path)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img);               axes[0].set_title(f"Kusurlu: {defect_type}")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Maske")
    axes[2].imshow(img); axes[2].imshow(mask, cmap="Reds", alpha=0.4); axes[2].set_title("Üst üste")
    for a in axes: a.axis("off")
    plt.show()

goster("leather","cut")

## AUROC
Model her görüntüye puan verir, eşik karar verir. AUROC = bozuğu sağlamdan ayırma başarısı (1.0 mükemmel, 0.5 yazı-tura).

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# 0 = sağlam, 1 = kusurlu
y_true = np.array([0]*22 + [1]*93)   # metal_nut test setiyle aynı sayılar

# Durum 1: model hiçbir şey bilmiyor (rastgele skor)
rastgele = rng.random(115)

# Durum 2: iyi bir model (kusurlulara daha yüksek skor)
iyi = np.where(y_true == 1, rng.normal(0.7, 0.15, 115), rng.normal(0.3, 0.15, 115))

for ad, skor in [("rastgele", rastgele), ("iyi model", iyi)]:
    fpr, tpr, _ = roc_curve(y_true, skor)
    plt.plot(fpr, tpr, label=f"{ad} (AUROC={roc_auc_score(y_true, skor):.2f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("FPR (yanlış alarm oranı)")
plt.ylabel("TPR (yakalanan kusur oranı)")
plt.legend(); plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

gercek = [0, 0, 0, 1, 0, 1, 1, 1]                      # 0 = sağlam, 1 = bozuk (A..H)
puan   = [0.10, 0.20, 0.35, 0.40, 0.50, 0.60, 0.80, 0.90]

print("AUROC:", roc_auc_score(gercek, puan))

## PyTorch ve tensörler
Tensör = sayı tablosu. Görüntüyü modele vermek için: permute (eksen sırası), /255 (0-1 aralığı), unsqueeze (batch).

In [ ]:
import torch
import cv2
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

def load(p):
    return cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)

In [ ]:
yol = next((ROOT/"metal_nut/train/good").glob("*.png"))
img = load(yol)

print("şekil:", img.shape)
print("en küçük / en büyük değer:", img.min(), img.max())

In [ ]:
t = torch.from_numpy(img)
print("ilk hali:", t.shape)

t = t.permute(2, 0, 1)
print("permute sonrası:", t.shape)

t = t.float() / 255.0
print("son hali:", t.shape, t.min().item(), t.max().item())

In [ ]:
batch = t.unsqueeze(0)
print(batch.shape)

In [ ]:
import torch
import cv2
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

## Dataset ve DataLoader
Dataset tek görüntüyü hazırlar (oku, boyutlandır, tensöre çevir, normalize et). DataLoader görüntüleri gruplar halinde modele taşır.

In [ ]:
class MVTecDataset(Dataset):
    def __init__(self, root, category, split="train", size=224):
        self.size = size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

        klasor = Path(root) / category / split
        self.dosyalar = sorted(klasor.rglob("*.png"))

    def __len__(self):
        return len(self.dosyalar)

    def __getitem__(self, i):
        yol = self.dosyalar[i]

        img = cv2.cvtColor(cv2.imread(str(yol)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.size, self.size))

        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        t = (t - self.mean) / self.std
        return t

In [ ]:
ds = MVTecDataset(ROOT, "metal_nut", split="train")
print("görüntü sayısı:", len(ds))

x = ds[0]
print("tek görüntü:", x.shape)

In [ ]:
loader = DataLoader(ds, batch_size=16, shuffle=False)

batch = next(iter(loader))
print("batch:", batch.shape)
print("kaç batch var:", len(loader))

## ResNet ile özellik çıkarma
ImageNet ile eğitilmiş resnet18 yüklenir. PatchCore'un kullanacağı ara katman (layer2, layer3) çıktılarının şekli incelenir.

In [ ]:
import torch
import torchvision

model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.eval()
print("model yüklendi")

In [ ]:
ds = MVTecDataset(ROOT, "metal_nut", split="train")
x = ds[0].unsqueeze(0)          # (1, 3, 224, 224) batch
print("giriş:", x.shape)

with torch.no_grad():
    cikis = model(x)
print("çıkış:", cikis.shape)

In [ ]:
ozellikler = {}

def kaydet(isim):
    def hook(modul, girdi, cikti):
        ozellikler[isim] = cikti
    return hook

model.layer1.register_forward_hook(kaydet("layer1"))
model.layer2.register_forward_hook(kaydet("layer2"))
model.layer3.register_forward_hook(kaydet("layer3"))
model.layer4.register_forward_hook(kaydet("layer4"))

with torch.no_grad():
    model(x)

for isim, f in ozellikler.items():
    print(isim, f.shape)

In [ ]:
import matplotlib.pyplot as plt

f = ozellikler["layer2"][0]          # (128, 28, 28)
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for i, ax in enumerate(axes):
    ax.imshow(f[i], cmap="viridis"); ax.axis("off"); ax.set_title(f"kanal {i}")
plt.show()